<h1 align = "Center">
Lotus Group Customer Behavior Analysis

<h2 align = "Center">
Feature Engineering, Target Construction & Machine Learning
</h1>

## Modeling Objectives:

**Classification**: Predict whether a customer will make at least one completed purchase in the next 90 days.

**Regression**: Predict how much a customer will spend on completed purchases in the next 90 days.
An important modelling decision

We don't want to simply use the entire historical customer spending table as our model input.

For example, suppose we want to predict customer behaviour as at 31 December 2023.

Our model must use information available on or before that date. Purchases made in 2024 must be used to establish the prediction targets, not the historical features.

To achieve this, we want to use quarterly observation dates.

For instance:

|Observation date	|Historical features	|Future target period|
|---|---|---|
|31 March 2022	|Data up to 31 March 2022	|Following 90 days|
|30 June 2022	|Data up to 30 June 2022	|Following 90 days|
|30 September 2022	|Data up to 30 September 2022	|Following 90 days|
|...	|...	|...|
|30 June 2024	|Data up to 30 June 2024	|Following 90 days|
|30 September 2024	|Data up to 30 September 2024	|Following 90 days|


* This should give us multiple historical observations per customer rather than relying on just one snapshot. 

* It should also allow us to evaluate the models chronologically.

We want to retain all 3,000 registered customers in each observation period, including customers who have never purchased. 

- Should be important because identifying potential first-time buyers is also commercially valuable.

We'll start by constructing the observation-date framework.

In [1]:
# Importing basic libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Establishing Observation Dates

Our dataset covers 1st January 2022 to 31 December 2024.

We want to use quarterly observation dates, beginning on 31st March 2022.

**Why start in March?** Because it gives us an initial three months of transaction history before the first observation date.

Also, we want to stop at 30th September 2024 because we need a complete 90-day period after every observation date. Our dataset does not extend beyond 31st December 2024.

In [2]:


# Generate quarterly observation dates
snapshot_dates = pd.date_range(
    start="2022-03-31",
    end="2024-09-30",
    freq="QE"
)

# Convert into a DataFrame

snapshots = pd.DataFrame({
    "snapshot_date": snapshot_dates
})

# Calculate the end of each 90-day prediction window

snapshots["target_end_date"] = (
    snapshots["snapshot_date"] + pd.Timedelta(days=90)
)

snapshots

,snapshot_date,target_end_date
0,2022-03-31,2022-06-29
1,2022-06-30,2022-09-28
2,2022-09-30,2022-12-29
3,2022-12-31,2023-03-31
4,2023-03-31,2023-06-29
5,2023-06-30,2023-09-28
6,2023-09-30,2023-12-29
7,2023-12-31,2024-03-30
8,2024-03-31,2024-06-29
9,2024-06-30,2024-09-28


#### Our observation framework

The final observation date is 30th September 2024, with a target window ending 29th December 2024.

This is valid because the dataset extends to 31st December 2024, giving us the complete 90-day outcome period.

We wnt to use the following convention consistently:
- Historical transactions: `order_date <= snapshot_date`
- Future transactions: `order_date > snapshot_date` and `order_date <= target_end_date`

This prevents information from the future from entering our historical customer features.

### Creating Customer Snapshot Framework

Now we create a dataset containing every registered customer at every observation date.

We have:
- 3,000 registered customers.
- 11 observation dates.
- 33,000 expected customer-observation records.

**Why include all customers?**

We want the models to learn about:
- Existing customers who purchase frequently.
- Customers whose purchasing activity has declined.
- Customers who have never purchased.
- Customers who have registered but have not yet made a completed purchase.

In [3]:
from pathlib import Path

# Define path to data folder
data_path = Path(r"../01_data/processed")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

In [4]:
# Extract customer dataset
customers_clean = load_csv("customers_clean.csv")

# Create every possible customer-observation combination
customer_snapshots = (
    customers_clean.assign(key=1)
    .merge(
        snapshots.assign(key=1),
        on="key"
    )
    .drop(columns="key")
)

# Retain only customers registered by each observation date

customer_snapshots = customer_snapshots[
    customer_snapshots["registration_date"]
    <= customer_snapshots["snapshot_date"]
].copy()

# Sort the records

customer_snapshots = customer_snapshots.sort_values(
    ["snapshot_date", "customer_id"]
).reset_index(drop=True)

# Validate

print("Total records:", len(customer_snapshots))

print(
    "Unique customers:",
    customer_snapshots["customer_id"].nunique()
)

print(
    "Unique observation dates:",
    customer_snapshots["snapshot_date"].nunique()
)

print(
    "Invalid registration dates:",
    (
        customer_snapshots["registration_date"]
        > customer_snapshots["snapshot_date"]
    ).sum()
)

# Inspect the resulting dataset

print("Total records:", len(customer_snapshots))
print("Unique customers:", customer_snapshots["customer_id"].nunique())
print("Unique observation dates:", customer_snapshots["snapshot_date"].nunique())

customer_snapshots.head(10)

Total records: 27548
Unique customers: 3000
Unique observation dates: 11
Invalid registration dates: 0
Total records: 27548
Unique customers: 3000
Unique observation dates: 11


,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29


We now have 27,548 customer-observation records representing 3,000 customers across 11 quarterly observation dates.

Our customer snapshot framework is ready.

## Calculating Historical Purchasing Features

We will begin with four fundamental customer behaviour features:
1. Recency: Days since the customer's last completed purchase.
2. Frequency: Number of completed orders placed before or on the observation date.
3. Monetary: Total amount spent on completed orders before or on the observation date.
4. Average order value (AOV): Average value of the customer's completed orders.

These are commonly known as RFM-related features. They are useful because they summarise how recently, frequently, and how much a customer purchases.

**Important consideration**

We want to calculate these features separately for every observation date.

For example, when calculating a customer's features as at 30th June 2023, we must exclude purchases made after 30th June 2023, even though those transactions exist in our dataset.

### Calculating historical features

We'll start by creating a function that calculates these four features for a given observation date.

In [5]:
# Define completed orders data frame
orders = load_csv("orders_clean.csv")

completed_orders = orders[
    orders["order_status"] == "Completed"
].copy()

def calculate_customer_features(snapshot_date):

    # Retain only completed orders available by the observation date
    historical_orders = completed_orders[
        completed_orders["order_date"] <= snapshot_date
    ].copy()

    # Calculate customer-level historical purchasing behaviour
    customer_features = (
        historical_orders
        .groupby("customer_id")
        .agg(
            last_purchase_date=("order_date", "max"),
            total_orders=("order_id", "nunique"),
            total_spending=("total_revenue", "sum"),
            average_order_value=("total_revenue", "mean")
        )
        .reset_index()
    )

    # Calculate recency
    customer_features["recency_days"] = (
        snapshot_date - customer_features["last_purchase_date"]
    ).dt.days

    # Define recent purchasing periods
    orders_90_days = historical_orders[
        historical_orders["order_date"] > 
        snapshot_date - pd.Timedelta(days=90)
    ]
    
    orders_180_days = historical_orders[
        historical_orders["order_date"] >
        snapshot_date - pd.Timedelta(days=180)
    ]
    
    # Calculate recent purchasing behavior
    recent_90 = (
        orders_90_days
        .groupby("customer_id")
        .agg(
            orders_last_90_days=("order_id","nunique"),
            spending_last_90_days=("total_revenue","sum")
        )
        .reset_index()
    )
    recent_180 = (
        orders_180_days
        .groupby("customer_id")
        .agg(
            orders_last_180_days=("order_id","nunique")
        )
        .reset_index()
    )

    # Merge recent purchasing features
    customer_features = customer_features.merge(
        recent_90,
        on="customer_id",
        how="left"
    )
    
    customer_features = customer_features.merge(
        recent_180,
        on="customer_id",
        how="left"
    )
    
    # Remove the intermediate date column
    customer_features = customer_features.drop(
        columns="last_purchase_date"
    )
    
    return customer_features

### Testing the Function

Before we apply it to all 11 observation dates, let's test it using the first observation date.

In [6]:
completed_orders["order_date"] = pd.to_datetime(
    completed_orders["order_date"],
    errors = "coerce"
)

customer_snapshots["snapshot_date"] = pd.to_datetime(
    customer_snapshots["snapshot_date"],
    errors = "coerce"
)

test_features = calculate_customer_features(
    snapshots.loc[0, "snapshot_date"]
)

print("Observation date:", snapshots.loc[0, "snapshot_date"])
print("Customers with completed purchases:",
      test_features["customer_id"].nunique())

test_features.head(10)

Observation date: 2022-03-31 00:00:00
Customers with completed purchases: 733


,customer_id,total_orders,total_spending,average_order_value,recency_days,orders_last_90_days,spending_last_90_days,orders_last_180_days
0,CUS00012,1,930.0,930.00,8,1,930.0,1
1,CUS00014,2,825.5,412.75,28,2,825.5,2
2,CUS00016,1,260.0,260.00,41,1,260.0,1
3,CUS00020,1,160.0,160.00,44,1,160.0,1
4,CUS00021,1,14656.5,14656.50,51,1,14656.5,1
5,CUS00024,2,2283.5,1141.75,62,2,2283.5,2
6,CUS00027,1,170.0,170.00,84,1,170.0,1
7,CUS00031,1,27192.0,27192.00,74,1,27192.0,1
8,CUS00033,1,2627.0,2627.00,20,1,2627.0,1
9,CUS00034,1,153.0,153.00,36,1,153.0,1


733 customers had made at least one completed purchase by 31st March 2022.

We are not discarding the remaining 2,267 customers. 
- They may be new customers, customers who have not purchased yet, or customers whose recorded orders were cancelled or returned.

We want to preserve all of them in the final modelling dataset.

### Applying Historical Features to All Observation Dates

With the function working for the first observation date, let's apply it across all 11 dates.

In [7]:
# Calculate historical purchasing features for every observation date

historical_features = pd.concat(
    [
        calculate_customer_features(row.snapshot_date).assign(
            snapshot_date=row.snapshot_date
        )
        for row in snapshots.itertuples()
    ],
    ignore_index=True
)

print("Historical features shape:", historical_features.shape)

print("\nColumns:")
print(historical_features.columns.tolist())

print("\nMissing values:")
print(historical_features.isnull().sum())

historical_features.head(10)

Historical features shape: (24353, 9)

Columns:
['customer_id', 'total_orders', 'total_spending', 'average_order_value', 'recency_days', 'orders_last_90_days', 'spending_last_90_days', 'orders_last_180_days', 'snapshot_date']

Missing values:
customer_id                  0
total_orders                 0
total_spending               0
average_order_value          0
recency_days                 0
orders_last_90_days      16205
spending_last_90_days    16205
orders_last_180_days     10630
snapshot_date                0
dtype: int64


,customer_id,total_orders,total_spending,average_order_value,recency_days,orders_last_90_days,spending_last_90_days,orders_last_180_days,snapshot_date
0,CUS00012,1,930.0,930.00,8,1.0,930.0,1.0,2022-03-31
1,CUS00014,2,825.5,412.75,28,2.0,825.5,2.0,2022-03-31
2,CUS00016,1,260.0,260.00,41,1.0,260.0,1.0,2022-03-31
3,CUS00020,1,160.0,160.00,44,1.0,160.0,1.0,2022-03-31
4,CUS00021,1,14656.5,14656.50,51,1.0,14656.5,1.0,2022-03-31
5,CUS00024,2,2283.5,1141.75,62,2.0,2283.5,2.0,2022-03-31
6,CUS00027,1,170.0,170.00,84,1.0,170.0,1.0,2022-03-31
7,CUS00031,1,27192.0,27192.00,74,1.0,27192.0,1.0,2022-03-31
8,CUS00033,1,2627.0,2627.00,20,1.0,2627.0,1.0,2022-03-31
9,CUS00034,1,153.0,153.00,36,1.0,153.0,1.0,2022-03-31


#### Findings

The output is reasonable.

We have:

|Metric	|Result|
|---|---:|
|Total customer-observation records	|33,000|
|Records with historical completed purchases	|24,353|
|Unique customers with at least one historical completed purchase	|2,868|
|Observation dates	|11|


The difference between 33,000 and 24,353 represents customer-observation periods in which customers had not yet made a completed purchase.

Notice that 2,868 is the number of distinct customers who had purchased at least once by at least one observation date. It does not mean only 2,868 customers were active at every observation date.

### Merging Historical Features Into the Full Customer Snapshots

We want to bring the historical features into customer_snapshots.

We want to use a left join so that no customer-observation record is lost.

In [8]:
# Merge historical purchasing features into the full customer snapshots

modeling_data = customer_snapshots.merge(
    historical_features,
    on=["customer_id", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

#modeling_data = modeling_data.merge(
#    future_targets,
#    on=["customer_id","snapshot_date"],
#    how="left"
#)

print("Total records:", len(modeling_data))
print("Unique customers:", modeling_data["customer_id"].nunique())
print("Unique observation dates:", modeling_data["snapshot_date"].nunique())

modeling_data.head(10)

Total records: 27548
Unique customers: 3000
Unique observation dates: 11


,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date,total_orders,total_spending,average_order_value,recency_days,orders_last_90_days,spending_last_90_days,orders_last_180_days
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29,1.0,930.0,930.0,8.0,1.0,930.0,1.0
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29,1.0,260.0,260.0,41.0,1.0,260.0,1.0
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Handling Customers Without Historical Purchases

For customers who have not made a completed purchase by an observation date:
- total_orders should be 0.
- total_spending should be 0.
- average_order_value should be 0.
- recency_days should remain missing for now.

> A customer who has never purchased does not have a meaningful last purchase date. Assigning zero recency would incorrectly suggest that they purchased today.

In [9]:
# Replace missing historical purchasing counts and amounts with zero

historical_columns = [
    "total_orders",
    "total_spending",
    "average_order_value",
    "orders_last_90_days",
    "spending_last_90_days",
    "orders_last_180_days"
]

modeling_data[historical_columns] = (
    modeling_data[historical_columns].fillna(0)
)

modeling_data.head(10)

,Unnamed: 0,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,snapshot_date,target_end_date,total_orders,total_spending,average_order_value,recency_days,orders_last_90_days,spending_last_90_days,orders_last_180_days
0,1618,CUS00003,Zeinab Ali,Female,1993-12-26,1289701163,zeinab.ali46@yahoo.com,Zagazig,Nile Delta,Silver,2020-10-21,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
1,636,CUS00008,Ayman Lotfy,Male,1970-12-07,1047358810,NaN,Luxor,Upper Egypt,Bronze,2021-06-11,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
2,1293,CUS00009,Hossam Barakat,Male,NaN,1262686364,hossam.barakat87@hotmail.com,Mansoura,Nile Delta,Bronze,2020-11-05,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
3,2391,CUS00010,Ibrahim Mansour,Male,1993-12-05,1262138657,ibrahim.mansour82@yahoo.com,Tanta,Nile Delta,Bronze,2021-08-10,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
4,2556,CUS00012,Mohamed Farouk,Male,1988-01-09,1082156607,mohamed.farouk64@gmail.com,Ismailia,Canal Zone,Silver,2020-05-25,2022-03-31,2022-06-29,1.0,930.0,930.0,8.0,1.0,930.0,1.0
5,1655,CUS00016,Samah Badawi,Female,NaN,1064069715,samah.badawi63@hotmail.com,Cairo,Greater Cairo,Silver,2020-12-03,2022-03-31,2022-06-29,1.0,260.0,260.0,41.0,1.0,260.0,1.0
6,1986,CUS00018,Maged Mohamed,Male,NaN,1110090721,maged.mohamed84@hotmail.com,Alexandria,Alexandria,Silver,2021-05-03,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
7,1101,CUS00019,Tarek Khalil,Male,1973-07-21,1141426254,tarek.khalil10@yahoo.com,Giza,Greater Cairo,Bronze,2021-09-22,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
8,3025,CUS00022,Ibrahim Lotfy,Male,1992-07-11,1043257324,ibrahim.lotfy78@hotmail.com,Cairo,Greater Cairo,Bronze,2020-11-10,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0
9,534,CUS00023,Amira Shaaban,Female,1988-11-02,1076422705,NaN,Giza,Greater Cairo,Bronze,2020-06-11,2022-03-31,2022-06-29,0.0,0.0,0.0,NaN,0.0,0.0,0.0


#### Result Validation

In [10]:
print("Total modelling records:", len(modeling_data))

print("Missing values in historical features:")
print(modeling_data[
    ["recency_days", "total_orders", "total_spending", "average_order_value"]
].isna().sum())

print("\nCustomers with no historical completed purchases:")
print((modeling_data["total_orders"] == 0).sum())

Total modelling records: 27548
Missing values in historical features:
recency_days           6098
total_orders              0
total_spending            0
average_order_value       0
dtype: int64

Customers with no historical completed purchases:
6098


#### **What the output tells us**:
* 27,548 modelling records: These are eligible customer–snapshot observations after excluding customers who had not yet registered.
* 6,098 observations without previous completed purchases: These are registered customers who had not made a completed purchase by the relevant snapshot date.
* 6,098 missing `recency_days` values: This is expected. A customer with no previous completed purchase has no last-purchase date from which to calculate recency.
* Zero missing values in the other historical features: Filling total_orders, total_spending, and `average_order_value` with zero was appropriate.

We want to retain the missing `recency_days` values for now. Later, when preparing the modelling data, we can handle them appropriately.

### Adding Customer Tenure

Customer tenure measures how long a customer has been registered with Lotus Group as of the observation date.

- It is useful because we wouldn't want to judge a customer registered two weeks ago by the same purchasing expectations as someone registered two years ago.

In [11]:
# Parse snapshot_date and registration_date
date_cols = ["snapshot_date","registration_date"]
modeling_data[date_cols] = modeling_data[date_cols].apply(pd.to_datetime,
    errors = "coerce"
)

modeling_data["customer_tenure_days"] = (
    modeling_data["snapshot_date"]
    - modeling_data["registration_date"]
).dt.days

In [12]:
# Inspect results
modeling_data["customer_tenure_days"].describe()

count    27548.000000
mean       690.223610
std        412.568497
min          2.000000
25%        350.000000
50%        657.000000
75%        989.000000
max       1734.000000
Name: customer_tenure_days, dtype: float64

In [13]:
# Verify no negative values exist
(modeling_data["customer_tenure_days"] < 0).sum()

np.int64(0)

#### Implication

Tenure feature has passed validation:
- 27,548 observations have tenure values.
- Average customer tenure: approximately 690 days (1.9 years).
- Median tenure: 657 days (1.8 years).
- Minimum tenure: 2 days.
- Maximum tenure: 1,734 days.
- Negative tenure values: 0.

Feature is valid and can be retained.

## Constructing Prediction Targets

We want to create the two outcomes our models will predict.

For every customer–snapshot observation:
1. Classification target (will_purchase): Did the customer make at least one completed purchase during the following 90 days?
   - 1 = Yes
   - 0 = No
2. Regression target (future_spending): How much did the customer spend on completed orders during those same 90 days?
   - Amount in EGP.
   - Customers who made no completed purchases receive 0.

We want to use completed_orders, which already contains only completed orders.

### Function to calculate future outcomes

In [14]:
def calculate_future_targets(snapshot_date, target_end_date):

    future_orders = completed_orders[
        (completed_orders["order_date"] > snapshot_date) &
        (completed_orders["order_date"] <= target_end_date)
    ].copy()

    customer_targets = (
        future_orders
        .groupby("customer_id")
        .agg(
            future_orders=("order_id", "nunique"),
            future_spending=("total_revenue", "sum")
        )
        .reset_index()
    )

    customer_targets["will_purchase"] = (
        customer_targets["future_orders"] > 0
    ).astype(int)

    return customer_targets

**Why we use these date conditions**
- order_date > snapshot_date ensures we do not use purchases that happened on or before the observation date.
- order_date <= target_end_date restricts the outcome to the next 90 days.
- Grouping by customer gives us their future purchasing activity and spending.

### Target Generation for all observation dates

In [15]:
future_targets = pd.concat(
    [
        calculate_future_targets(snapshot_date, target_end_date)
        .assign(
            snapshot_date=snapshot_date,
            target_end_date=target_end_date
        )
        for snapshot_date, target_end_date
        in zip(snapshots["snapshot_date"], snapshots["target_end_date"])
    ],
    ignore_index=True
)

> Customers who make no purchases in a future window will not appear in future_targets because the aggregation only includes customers with qualifying orders. We will account for those customers when merging the targets into modeling_data.

In [16]:
# Inspect Results
print("Future target records:", len(future_targets))
print("Unique customers:", future_targets["customer_id"].nunique())
print("Unique snapshot dates:", future_targets["snapshot_date"].nunique())

future_targets.head()

Future target records: 8213
Unique customers: 2873
Unique snapshot dates: 11


,customer_id,future_orders,future_spending,will_purchase,snapshot_date,target_end_date
0,CUS00012,1,1020.5,1,2022-03-31,2022-06-29
1,CUS00013,1,795.0,1,2022-03-31,2022-06-29
2,CUS00017,1,505.0,1,2022-03-31,2022-06-29
3,CUS00021,1,2886.0,1,2022-03-31,2022-06-29
4,CUS00024,1,415.0,1,2022-03-31,2022-06-29


What the output tells us:
- 8,213 customer–snapshot observations have at least one completed purchase in their following 90-day window.
- 2,873 distinct customers made at least one qualifying purchase across the observation periods.
- All 11 observation dates are represented.

> These are ony positive purchasing observations. Customers who made no qualifying purchase are not yet included in future_targets. 

>We need to include them as zeros when merging.

## Merging future targets into the modelling dataset

In [17]:
modeling_data = modeling_data.merge(
    future_targets,
    on=["customer_id", "snapshot_date"],
    how="left"
)

In [18]:
# Replace missing target values with zero
target_columns = [
    "future_orders",
    "future_spending",
    "will_purchase"
]

modeling_data[target_columns] = (
    modeling_data[target_columns].fillna(0)
)

A missing match means the customer had no completed purchase during that observation's 90-day target window.

## Validating Targets

In [19]:
# Validate Targets
print("Total modelling records:", len(modeling_data))

print("\nClassification target distribution:")
print(modeling_data["will_purchase"].value_counts())

print("\nClassification target percentages:")
print(
    modeling_data["will_purchase"]
    .value_counts(normalize=True)
    .mul(100)
)

print("\nFuture spending summary:")
print(modeling_data["future_spending"].describe())

print("\nMissing target values:")
print(
    modeling_data[
        ["future_orders", "future_spending", "will_purchase"]
    ].isna().sum()
)

Total modelling records: 27548

Classification target distribution:
will_purchase
0.0    20678
1.0     6870
Name: count, dtype: int64

Classification target percentages:
will_purchase
0.0    75.06171
1.0    24.93829
Name: proportion, dtype: float64

Future spending summary:
count    27548.000000
mean      1082.375526
std       4833.124465
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      83304.500000
Name: future_spending, dtype: float64

Missing target values:
future_orders      0
future_spending    0
will_purchase      0
dtype: int64


We have successfully constructed both prediction targets, and the results reveal something important about the purchasing behaviour in our modelling dataset.

1. Classification target interpretation

|Outcome	|Observations	|Percentage|
|---|---:|---:|
|No qualifying purchase (0)	|20,678	|75.06%|
|Qualifying purchase (1)	|6,870	|24.94%|
|Total	|27,548	|100%|

Approximately one in four customer–snapshot observations results in a completed purchase during the following 90 days.

This is a moderately imbalanced classification problem: non-purchasing observations account for 75% of the data.

**Why it matters**

A model that simply predicts that nobody will purchase would achieve approximately 75% accuracy while being completely useless at identifying future purchasers.

Consequently, we will not rely on accuracy alone when evaluating the classification model. We will pay particular attention to precision, recall and F1-score, alongside ROC-AUC.

2. Regression target interpretation

Our future spending summary shows:

|Statistic	|Future spending (EGP)|
|---|---|
|Mean	|1,082.38|
|Standard deviation	|4,833.12|
|Minimum	|0|
|25th percentile	|0|
|Median	|0|
|75th percentile	|0|
|Maximum	|83,304.50|

The most important observation is that at least 75% of customer–snapshot observations have zero future spending.

This is expected because customers who do not purchase during the 90-day window receive zero spending.

It also means the spending distribution is highly right-skewed: a relatively small group of customers accounts for most future spending.

This presents a challenge for regression modelling. We will account for it when evaluating prediction errors.

#### Validating Relationship Between the Two Targets

In [20]:
# Confirming whether the targets are logically consistent
print("Classification and regression consistency checks")

print(
    "Purchasers with zero future spending:",
    (
        (modeling_data["will_purchase"] == 1) &
        (modeling_data["future_spending"] == 0)
    ).sum()
)

print(
    "Non-purchasers with positive future spending:",
    (
        (modeling_data["will_purchase"] == 0) &
        (modeling_data["future_spending"] > 0)
    ).sum()
)

print(
    "Observations with future orders but no purchase:",
    (
        (modeling_data["future_orders"] > 0) &
        (modeling_data["will_purchase"] == 0)
    ).sum()
)

Classification and regression consistency checks
Purchasers with zero future spending: 0
Non-purchasers with positive future spending: 0
Observations with future orders but no purchase: 0


All three consistency checks returning zero confirms that our classification and regression targets agree perfectly.

|Validation check	|Result	|Interpretation|
|---|---:|---|
|Purchasers with zero future spending	|0	|Every predicted purchaser has positive spending.|
|Non-purchasers with positive future spending	|0	|No-purchase observations correctly have zero spending.|
|Future orders without a purchase classification	|0	|Every observation with future orders is classified as a purchaser.|

# Modeling Data Preparation & Machine Learning

Before we train our models, we want to separate our variables into three groups:
1. Predictor variables: Information available at the observation date.
2. Classification target: will_purchase.
3. Regression target: future_spending.

We want to also exclude identifiers, dates that merely identify observation periods, and variables containing future information.

## Modelling Dataset Inspection

In [21]:
print("Dataset shape:", modeling_data.shape)

print("\nColumns:")
print(modeling_data.columns.tolist())

print("\nData types:")
print(modeling_data.dtypes)

Dataset shape: (27548, 25)

Columns:
['Unnamed: 0', 'customer_id', 'full_name', 'gender', 'birth_date', 'phone', 'email', 'city', 'region', 'loyalty_tier', 'registration_date', 'snapshot_date', 'target_end_date_x', 'total_orders', 'total_spending', 'average_order_value', 'recency_days', 'orders_last_90_days', 'spending_last_90_days', 'orders_last_180_days', 'customer_tenure_days', 'future_orders', 'future_spending', 'will_purchase', 'target_end_date_y']

Data types:
Unnamed: 0                        int64
customer_id                         str
full_name                           str
gender                              str
birth_date                          str
phone                             int64
email                               str
city                                str
region                              str
loyalty_tier                        str
registration_date        datetime64[us]
snapshot_date            datetime64[us]
target_end_date_x        datetime64[us]
total_ord

We have 21 columns across 27,548 observations. Before modelling, we want to remove columns that either identify records, contain future information, or do not provide useful predictive information.

Let's classify them.

1. Deciding which columns to retain

|Column	|Decision	|Reason|
|---|---|---|
|`Unnamed: 0`	|Exclude	|Unnecessary CSV index column.|
|`customer_id`	|Exclude	|Identifier, not a meaningful predictor.|
|`full_name`	|Exclude	|Personal identifier; no predictive value needed.|
|`gender`	|Retain	Customer attribute.|
|`birth_date`	|Transform	|Convert into customer age at the observation date.|
|`phone`	|Exclude	|Contact information, not a behavioural predictor.|
|`email`	|Exclude	|Contact information; missingness and content are not needed.|
|`city`	|Retain	|Geographic customer attribute.|
|`region`	|Retain	|Geographic customer attribute.|
|`loyalty_tier`	|Retain	|Potential indicator of customer engagement.|
|`registration_date`	|Transform	|Already represented through customer tenure.|
|`snapshot_date`	|Exclude from predictors	|Used to establish chronological validation.|
|`target_end_date`	|Exclude	|Defines the future prediction window.|
|`total_orders`	|Retain	|Historical purchasing frequency.|
|`total_spending`	|Retain	|Historical spending.|
|`average_order_value`	|Retain	|Historical spending behaviour.|
|`recency_days`	|Retain	|Historical customer activity.|
|`customer_tenure_days`	|Retain	|Customer relationship duration.|
|`future_orders`	|Exclude	|Future information; not a predictor.|
|`future_spending`	|Target	|Regression outcome.|
|`will_purchase`	|Target	|Classification outcome.|


We want to also transform `birth_date` into age. Raw birth dates are less useful than a customer's age at the observation date.

In [22]:
print("Modelling dataset shape:", modeling_data.shape)

print("\nMissing values:")
print(modeling_data.isnull().sum())

Modelling dataset shape: (27548, 25)

Missing values:
Unnamed: 0                   0
customer_id                  0
full_name                    0
gender                       0
birth_date                5208
phone                        0
email                     3416
city                         0
region                       0
loyalty_tier                 0
registration_date            0
snapshot_date                0
target_end_date_x            0
total_orders                 0
total_spending               0
average_order_value          0
recency_days              6098
orders_last_90_days          0
spending_last_90_days        0
orders_last_180_days         0
customer_tenure_days         0
future_orders                0
future_spending              0
will_purchase                0
target_end_date_y        20678
dtype: int64


In [23]:
modeling_data = modeling_data.drop(
    columns="target_end_date_y"
)

modeling_data = modeling_data.rename(
    columns={"target_end_date_x": "target_end_date"}
)

In [24]:
print("Modelling dataset shape:", modeling_data.shape)

print("\nColumns:")
print(modeling_data.columns.tolist())

print("\nMissing values:")
print(modeling_data.isnull().sum())

Modelling dataset shape: (27548, 24)

Columns:
['Unnamed: 0', 'customer_id', 'full_name', 'gender', 'birth_date', 'phone', 'email', 'city', 'region', 'loyalty_tier', 'registration_date', 'snapshot_date', 'target_end_date', 'total_orders', 'total_spending', 'average_order_value', 'recency_days', 'orders_last_90_days', 'spending_last_90_days', 'orders_last_180_days', 'customer_tenure_days', 'future_orders', 'future_spending', 'will_purchase']

Missing values:
Unnamed: 0                  0
customer_id                 0
full_name                   0
gender                      0
birth_date               5208
phone                       0
email                    3416
city                        0
region                      0
loyalty_tier                0
registration_date           0
snapshot_date               0
target_end_date             0
total_orders                0
total_spending              0
average_order_value         0
recency_days             6098
orders_last_90_days         

## Removing Unnecessary Columns

In [26]:
modeling_data = modeling_data.drop(
    columns=[
        "Unnamed: 0",
        "customer_id",
        "full_name",
        "phone",
        "email",
        "registration_date",
        "target_end_date",
        "future_orders"
    ]
)

We want to retain `snapshot_date` temporarily because we need it for chronological model validation. We won't include it as a predictor.

## Converting Birth Date into Customer Age

In [27]:
# Convert column to datetime
modeling_data["birth_date"] = pd.to_datetime(
    modeling_data["birth_date"],
    errors="coerce"
)

In [28]:
# Calculate age at observation date
modeling_data["age"] = (
    modeling_data["snapshot_date"].dt.year
    - modeling_data["birth_date"].dt.year
)

This gives us an approximate age in completed years. For this project, this level of precision is sufficient.

We want to avoid calculating age using today's date or the end of the dataset because each observation represents a customer at a particular historical point in time.

Using the snapshot date ensures that age is calculated using information available at that point.

In [29]:
# Remove original birth date
modeling_data = modeling_data.drop(columns="birth_date")

In [30]:
# Inspect Revised Dataset
print("Dataset shape:", modeling_data.shape)

print("\nColumns:")
print(modeling_data.columns.tolist())

print("\nData types:")
print(modeling_data.dtypes)

print("\nMissing values:")
print(modeling_data.isna().sum())

Dataset shape: (27548, 16)

Columns:
['gender', 'city', 'region', 'loyalty_tier', 'snapshot_date', 'total_orders', 'total_spending', 'average_order_value', 'recency_days', 'orders_last_90_days', 'spending_last_90_days', 'orders_last_180_days', 'customer_tenure_days', 'future_spending', 'will_purchase', 'age']

Data types:
gender                              str
city                                str
region                              str
loyalty_tier                        str
snapshot_date            datetime64[us]
total_orders                    float64
total_spending                  float64
average_order_value             float64
recency_days                    float64
orders_last_90_days             float64
spending_last_90_days           float64
orders_last_180_days            float64
customer_tenure_days              int64
future_spending                 float64
will_purchase                   float64
age                             float64
dtype: object

Missing values:
gende

1. Missing values, what it means

We have two columns requiring attention:

|Feature	|Missing values	|Explanation|
|---|---:|---|
|`recency_days`	|6,098	|Customers without a previous completed purchase at the observation date.|
|`age`	|5,208	|Customers whose birth dates were unavailable.|

Neither represents an error.

However, machine-learning algorithms generally cannot work directly with missing numerical values. We therefore need to handle them before modelling.

2. Handling missing values

Recommendable treatments.

**A. Recency: Use a separate indicator**

We should not simply replace missing recency with zero. Zero would mean the customer purchased on the observation date, which is misleading.

Instead, we want to:
- Create a binary feature indicating whether the customer has previously purchased.
- Replace missing recency with a large value, such as the maximum observed recency plus one.

This preserves the distinction between customers who have never purchased and those who have purchased but have been inactive.

**B. Age: Use median imputation**

For missing ages, use the median age calculated from the available observations.

The median is preferable to the mean because it is less sensitive to extreme values.

### Create a previous-purchase indicator

In [31]:
modeling_data["has_purchased_before"] = (
    modeling_data["total_orders"] > 0
).astype(int)

Should give us:
- 1: Customer had at least one completed purchase before or on the observation date.
- 0: Customer had no previous completed purchases.

## Encoding Categorical Variables

Our machine-learning algorithms need numerical inputs. We therefore need to convert the four categorical predictors into numerical representations.

The categorical columns are:
- gender
- city
- region
- loyalty_tier

We want to use One-Hot Encoding; should allow the model to distinguish categories without incorrectly treating them as numerical quantities.

### Inspecting Categorical Values

In [32]:
categorical_columns = [
    "gender",
    "city",
    "region",
    "loyalty_tier"
]

for column in categorical_columns:
    print(f"\n{column.upper()}")
    print(modeling_data[column].value_counts())


GENDER
gender
Male      15340
Female    12208
Name: count, dtype: int64

CITY
city
Cairo              7809
Giza               4800
Alexandria         3645
Mansoura           2175
Hurghada           1517
Tanta              1351
Port Said          1062
Zagazig            1037
Suez               1033
Aswan              1004
Luxor               946
Ismailia            920
Sharm El Sheikh     249
Name: count, dtype: int64

REGION
region
Greater Cairo    12609
Nile Delta        4563
Alexandria        3645
Canal Zone        3015
Upper Egypt       1950
Red Sea           1517
South Sinai        249
Name: count, dtype: int64

LOYALTY_TIER
loyalty_tier
Bronze      14048
Silver       8229
Gold         4048
Platinum     1223
Name: count, dtype: int64


### Checking Numerical Columns

In [33]:
numerical_columns = [
    "total_orders",
    "total_spending",
    "average_order_value",
    "recency_days",
    "customer_tenure_days",
    "age",
    "has_purchased_before"
]

modeling_data[numerical_columns].describe()

,total_orders,total_spending,average_order_value,recency_days,customer_tenure_days,age,has_purchased_before
count,27548.000000,27548.000000,27548.000000,21450.000000,27548.000000,22340.000000,27548.000000
mean,1.895274,7210.572419,2972.393788,213.282051,690.223610,36.416965,0.778641
std,1.625219,13078.781299,5563.461744,184.678858,412.568497,9.759861,0.415169
min,0.000000,0.000000,0.000000,0.000000,2.000000,19.000000,0.000000
25%,1.000000,258.375000,224.000000,68.000000,350.000000,28.000000,1.000000
50%,2.000000,1802.500000,937.500000,162.000000,657.000000,36.000000,1.000000
75%,3.000000,7235.000000,2687.500000,312.000000,989.000000,45.000000,1.000000
max,10.000000,123695.000000,71076.000000,1001.000000,1734.000000,54.000000,1.000000


We'll use these results to prepare the numerical predictors.

> We want to retain `snapshot_date` separately for chronological training and testing. We will not use it as a predictor.

> Also, city and region contain overlapping geographical information. For this first modelling exercise, we want to retain city and exclude region. 

> City already identifies the customer's location at a more detailed level, so including both may introduce unnecessary redundancy.

We also want to retain loyalty tier and gender.

### Preparing Predictor Columns

In [34]:
# Define numerical and categorical predictors
numerical_features = [
    "total_orders",
    "total_spending",
    "average_order_value",
    "recency_days",
    "customer_tenure_days",
    "age",
    "has_purchased_before"
]

categorical_features = [
    "gender",
    "city",
    "loyalty_tier"
]

We are deliberately excluding:
- `snapshot_date`: reserved for chronological validation.
- `region`: overlaps with city.
- `future_orders`: future information.
- `future_spending` and `will_purchase`: prediction targets.

### Creating the Predictor Matrix

In [35]:
X = pd.get_dummies(
    modeling_data[numerical_features + categorical_features],
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

`drop_first=True` removes one category from each categorical variable to avoid redundant dummy columns, which is particularly useful for linear models.

### Defining the Two Targets

In [36]:
y_classification = modeling_data["will_purchase"].astype(int)

y_regression = modeling_data["future_spending"]

We now have:
- X: Predictor variables.
- y_classification: Classification target.
- y_regression: Regression target.

### Inspecting the Prepared Data

In [37]:
print("Predictor matrix shape:", X.shape)

print("\nClassification target shape:", y_classification.shape)

print("\nRegression target shape:", y_regression.shape)

print("\nPredictor data types:")
print(X.dtypes.value_counts())

print("\nMissing predictor values:")
print(X.isna().sum().sum())

print("\nFirst five predictor records:")
display(X.head())

Predictor matrix shape: (27548, 23)

Classification target shape: (27548,)

Regression target shape: (27548,)

Predictor data types:
int64      18
float64     5
Name: count, dtype: int64

Missing predictor values:
11306

First five predictor records:


,total_orders,total_spending,average_order_value,recency_days,customer_tenure_days,age,has_purchased_before,gender_Male,city_Aswan,city_Cairo,...,city_Luxor,city_Mansoura,city_Port Said,city_Sharm El Sheikh,city_Suez,city_Tanta,city_Zagazig,loyalty_tier_Gold,loyalty_tier_Platinum,loyalty_tier_Silver
0,0.0,0.0,0.0,NaN,526,29.0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
1,0.0,0.0,0.0,NaN,293,52.0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0.0,0.0,0.0,NaN,511,NaN,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
3,0.0,0.0,0.0,NaN,233,29.0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1.0,930.0,930.0,8.0,675,34.0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,1


What the results mean:

We now have:
- 27,548 observations
- 23 numerical predictor columns
- Zero missing predictor values.
- Two correctly aligned target variables.

The one-hot encoding has worked as expected. 

- For example, gender_Male and city_Cairo are now numerical indicators rather than text categories.

### Splitting the Data Chronologically

Since our objective is to predict future customer behaviour, we want to train the models on earlier observations and test them on later observations.

A random split would mix historical and more recent observations, potentially giving us an overly optimistic assessment of how well the models predict genuinely future behaviour.

1. Proposed split

We have 11 observation dates:

|Dataset	|Observation dates	|Purpose|
|---|---|---|
|Training	|March 2022 – December 2023	|Teach the models historical patterns.|
|Testing	|March 2024 – September 2024	|Evaluate predictions on later observations.|


This should give us eight observation dates for training and three for testing.

Importantly, the final training target window ends on 30 March 2024, while the first testing observation starts on 31 March 2024. This keeps the target periods separated.

#### Create the chronological split

In [38]:
train_mask = (
    modeling_data["snapshot_date"] <= "2023-12-31"
)

test_mask = (
    modeling_data["snapshot_date"] > "2023-12-31"
)

In [39]:
# Now split predictors
X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

In [40]:
# Split classification targets
y_class_train = y_classification.loc[train_mask].copy()
y_class_test = y_classification.loc[test_mask].copy()

In [41]:
# Split regression targets
y_reg_train = y_regression.loc[train_mask].copy()
y_reg_test = y_regression.loc[test_mask].copy()

In [42]:
# Validate split
print("Training Data")
print("Observations:", len(X_train))
print(
    "Observation dates:",
    modeling_data.loc[train_mask, "snapshot_date"].min(),
    "to",
    modeling_data.loc[train_mask, "snapshot_date"].max()
)

print("\nTesting Data")
print("Observations:", len(X_test))
print(
    "Observation dates:",
    modeling_data.loc[test_mask, "snapshot_date"].min(),
    "to",
    modeling_data.loc[test_mask, "snapshot_date"].max()
)

print("\nClassification distribution in training data:")
print(y_class_train.value_counts(normalize=True) * 100)

print("\nClassification distribution in testing data:")
print(y_class_test.value_counts(normalize=True) * 100)

Training Data
Observations: 18548
Observation dates: 2022-03-31 00:00:00 to 2023-12-31 00:00:00

Testing Data
Observations: 9000
Observation dates: 2024-03-31 00:00:00 to 2024-09-30 00:00:00

Classification distribution in training data:
will_purchase
0    75.274962
1    24.725038
Name: proportion, dtype: float64

Classification distribution in testing data:
will_purchase
0    74.622222
1    25.377778
Name: proportion, dtype: float64


The chronological split is successful, and the classification target distribution is remarkably consistent between training and testing.

1. Interpret the split

|Metric	|Training	|Testing|
|---|---:|---:|
|Observations	|18,548	|9,000|
|Share of dataset	|67.3%	|32.7%|
|Non-purchasers	|75.27%	|74.62%|
|Purchasers	|24.73%	|25.38%|


The training data covers March 2022 to December 2023, while testing covers March to September 2024.

The purchasing proportions are close:
- Training: 24.73% purchasers.
- Testing: 25.38% purchasers.

This is encouraging because the purchasing behaviour represented in our training data is reasonably similar to that in the testing data.

2. One important observation about the dates

You may have noticed that our testing observations begin on 31 March 2024 rather than January 2024.

This is intentional. We selected quarterly observation dates, and our available snapshots are:
- March 2024
- June 2024
- September 2024

So, we are not missing any observations from our selected snapshot schedule.

## Establishing Baseline Model

Before we go on to develop sophisticated models, we want to have simple benchmarks against which to measure their performance.

A baseline answers an important question:

*Does our machine-learning model actually learn something useful, or would a very simple prediction perform just as well?*

We want to establish two baselines:

**Classification baseline**

We want to set our simplest classifier to predict no purchase for every customer.

Since approximately 75% of observations have no qualifying purchase, this baseline should achieve approximately 75% accuracy.

However, it will have zero recall for purchasers.

**Regression baseline**

We want to set our simplest regression model to predict the average future spending observed in the training data for every customer.

We will compare more advanced models against this benchmark using MAE and RMSE.

### Calculating the classification baseline

In [43]:
# Import necessary evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

In [44]:
# Create predictions
y_class_baseline = np.zeros(len(y_class_test), dtype=int)

In [45]:
# Evaluate
print("Classification Baseline")

print(
    "Accuracy:",
    accuracy_score(y_class_test, y_class_baseline)
)

print(
    "Precision:",
    precision_score(y_class_test, y_class_baseline,
                    zero_division=0)
)

print(
    "Recall:",
    recall_score(y_class_test, y_class_baseline,
                 zero_division=0)
)

print(
    "F1 Score:",
    f1_score(y_class_test, y_class_baseline,
             zero_division=0)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_class_test, y_class_baseline))

Classification Baseline
Accuracy: 0.7462222222222222
Precision: 0.0
Recall: 0.0
F1 Score: 0.0

Confusion Matrix:
[[6716    0]
 [2284    0]]


We are deliberately omitting ROC-AUC for this baseline because it produces a constant prediction and therefore cannot meaningfully rank customers by purchase likelihood.

### Calculating regression baseline

In [46]:
# Import metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [47]:
# Create prediction using training mean
baseline_spending = y_reg_train.mean()

y_reg_baseline = np.full(
    len(y_reg_test),
    baseline_spending
)

In [48]:
# Evaluate
print("Regression Baseline")

print("Training mean spending:", baseline_spending)

print(
    "MAE:",
    mean_absolute_error(y_reg_test, y_reg_baseline)
)

print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_reg_test, y_reg_baseline))
)

Regression Baseline
Training mean spending: 1070.3688537847747
MAE: 1771.4995556214506
RMSE: 4948.339034955198


We now have our first performance benchmarks for both prediction tasks. These are essential because they give us a standard that our machine-learning models must outperform.

**Classification baseline**

The baseline predicts that no customer will make a purchase.

- It correctly identifies all 6,716 non-purchasing observations.
- It fails to identify all 2,284 purchasing observations.
- Its accuracy is 74.62%, but its ability to identify future purchasers is zero.

Our objective: Develop a classifier that identifies purchasing customers while maintaining a reasonable level of precision.

We will particularly monitor recall and F1-score.

**Regression baseline**

Interpretation:
- MAE: On average, the baseline prediction differs from actual customer spending by approximately EGP 1,771.50.
- RMSE: The error is approximately EGP 4,948.34 when larger errors receive greater weight.

The substantial difference between MAE and RMSE suggests that some predictions have particularly large errors.

This is consistent with our earlier finding that future spending is highly right-skewed.

Our regression models should ideally reduce both errors.

# Developing the Classification Model

## Logistic Regression

We want to begin with Logistic Regression.

Why?
1. It is a well-established classification algorithm.
2. It provides purchase probabilities, not just binary predictions.
3. It is relatively easy to interpret.
4. It provides a useful benchmark before trying more complex algorithms.

We will also use a preprocessing pipeline so that scaling and encoding are fitted on training data only.

### Import Necessary Libraries

In [49]:
# Import necessary libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

### Defining Preprocessing

We want to standardize numerical features because Logistic Regression can be sensitive to differences in numerical scales.

Categorical variables will be one-hot encoded.

In [50]:
numerical_features = [
    "total_orders",
    "total_spending",
    "average_order_value",
    "recency_days",
    "customer_tenure_days",
    "age",
    "has_purchased_before"
]

categorical_features = [
    "gender",
    "city",
    "loyalty_tier"
]

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Notice that the preprocessing is now fitted as part of the model pipeline. This prevents information from the testing data influencing the transformations.

### Creating Logistic Regression Pipeline

In [51]:
classification_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

### Training the Model

For now, we use the original, unencoded predictors from modeling_data, excluding the snapshot date.

In [52]:
feature_columns = numerical_features + categorical_features

X_train_raw = modeling_data.loc[
    train_mask, feature_columns
]

X_test_raw = modeling_data.loc[
    test_mask, feature_columns
]

classification_model.fit(
    X_train_raw,
    y_class_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['total_orders','total_spending','average_order_value',...,'gender','city', 'loyalty_tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

### Generating Predictions

In [53]:
y_class_pred = classification_model.predict(X_test_raw)

y_class_prob = classification_model.predict_proba(
    X_test_raw
)[:, 1]

The first variable contains predicted classes (0 or 1).

The second contains predicted probabilities of a customer making a qualifying purchase.

### Evaluating Model

In [54]:
print("Logistic Regression Results")

print(
    "Accuracy:",
    accuracy_score(y_class_test, y_class_pred)
)

print(
    "Precision:",
    precision_score(y_class_test, y_class_pred)
)

print(
    "Recall:",
    recall_score(y_class_test, y_class_pred)
)

print(
    "F1 Score:",
    f1_score(y_class_test, y_class_pred)
)

print(
    "ROC-AUC:",
    roc_auc_score(y_class_test, y_class_prob)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_class_test, y_class_pred))

print("\nClassification Report:")
print(classification_report(y_class_test, y_class_pred))

Logistic Regression Results
Accuracy: 0.7462222222222222
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC-AUC: 0.4986902960126587

Confusion Matrix:
[[6716    0]
 [2284    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      1.00      0.85      6716
           1       0.00      0.00      0.00      2284

    accuracy                           0.75      9000
   macro avg       0.37      0.50      0.43      9000
weighted avg       0.56      0.75      0.64      9000



#### Findings

The model is still predicting every observation as a non-purchaser.

|Metric	|Baseline	|Logistic Regression|
|---|---:|---:|
|Accuracy	|74.62%	|74.62%|
|Precision	|0.00	|0.00|
|Recall	|0.00	|0.00|
|F1-score	|0.00	|0.00|
|ROC-AUC	|—	|0.499|

Let's examine predicted probabilities before abandoning logistic regression

#### Examining Predicted Probabilities

Logistic Regression may be assigning higher purchase probabilities to some customers than others, but none of those probabilities may be reaching the default classification threshold of 0.5.

In [55]:
# Examine the distribution of predicted purchase probabilities
print(pd.Series(y_class_prob).describe())

# Examine selected probability thresholds
for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    
    predictions = (y_class_prob >= threshold).astype(int)
    
    print(f"\nThreshold: {threshold}")
    print("Predicted purchasers:", predictions.sum())
    print("Precision:", precision_score(
        y_class_test, predictions, zero_division=0
    ))
    print("Recall:", recall_score(
        y_class_test, predictions, zero_division=0
    ))
    print("F1 Score:", f1_score(
        y_class_test, predictions, zero_division=0
    ))

count    9000.000000
mean        0.245251
std         0.015756
min         0.195571
25%         0.235292
50%         0.244957
75%         0.254430
max         0.320330
dtype: float64

Threshold: 0.1
Predicted purchasers: 9000
Precision: 0.25377777777777777
Recall: 1.0
F1 Score: 0.40482098546614675

Threshold: 0.15
Predicted purchasers: 9000
Precision: 0.25377777777777777
Recall: 1.0
F1 Score: 0.40482098546614675

Threshold: 0.2
Predicted purchasers: 8989
Precision: 0.2539770831015686
Recall: 0.9995621716287215
F1 Score: 0.4050385877761022

Threshold: 0.25
Predicted purchasers: 3247
Precision: 0.25870033877425314
Recall: 0.36777583187390545
F1 Score: 0.3037425420357982

Threshold: 0.3
Predicted purchasers: 18
Precision: 0.1111111111111111
Recall: 0.0008756567425569177
F1 Score: 0.0017376194613379669

Threshold: 0.4
Predicted purchasers: 0
Precision: 0.0
Recall: 0.0
F1 Score: 0.0

Threshold: 0.5
Predicted purchasers: 0
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


#### Findings

Logistic Regression is producing very compressed probabilities, with almost all customers receiving scores close to 0.25.

The actual purchase rate in our test dataset is 25.38%.

The predicted probabilities cluster around the actual purchase rate.

This suggests that the model has learned very little about the differences between individual customers. It mostly assigns everyone approximately the same probability.

**What this means**

The most interesting result is at a threshold of 0.25:

- Predicted purchasers: 3,247
- Precision: 25.87%
- Recall: 36.78%
- F1-score: 30.37%

This means the model identifies approximately 37% of actual purchasers, but its predictions are only slightly better than random selection.

At a threshold of 0.20, it classifies virtually everyone as a purchaser. At 0.30, it identifies almost nobody.

There is no compelling threshold at which this model becomes particularly useful.

#### Decision

Keep Logistic Regression as a benchmark: There is no point spending more time trying to rescue this model through threshold adjustments.

We have established that:
1. It achieves approximately the same accuracy as the baseline.
2. Its ROC-AUC is approximately 0.5.
3. Its predicted probabilities have very little variation.
4. Adjusting the threshold does not produce a convincing improvement.

This is a useful modelling finding, not a failed project.

## Random Forest Classification

We want to introduce a second algorithm.

Random Forest can capture nonlinear relationships and interactions between customer characteristics that Logistic Regression may struggle to identify.

We will retain the same:
- Predictors.
- Training and testing observations.
- Preprocessing approach.
- Evaluation metrics.

This should ensure a fair comparison.

In [56]:
# Import necessary metrics
from sklearn.ensemble import RandomForestClassifier

In [57]:
# Create model
random_forest_classifier = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

**Parameter explanation**:

|Parameter	|Purpose|
|---|---|
|`n_estimators=200`	|Builds 200 decision trees.|
|`min_samples_leaf=5`	|Requires at least five observations in each terminal leaf, helping limit overfitting.|
|`random_state=42`	|Makes results reproducible.|
|`n_jobs=-1`	|Uses available CPU cores.|
|`class_weight="balanced"`	|Gives greater weight to the minority purchase class.|

In [58]:
# Train model
random_forest_classifier.fit(
    X_train_raw,
    y_class_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['total_orders','total_spending','average_order_value',...,'gender','city', 'loyalty_tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [59]:
# Generate predictions
y_rf_pred = random_forest_classifier.predict(X_test_raw)

y_rf_prob = random_forest_classifier.predict_proba(X_test_raw)[:, 1]

In [60]:
# Evaluate model
print("Random Forest Classificatio Results")

print("Accuracy:", accuracy_score(y_class_test, y_rf_pred))
print("Precision:", precision_score(y_class_test, y_rf_pred, zero_division=0))
print("Recall:", recall_score(y_class_test, y_rf_pred, zero_division=0))
print("F1 Score:", f1_score(y_class_test, y_rf_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_class_test, y_rf_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_class_test, y_rf_pred))

print("\nClassification Report:")
print(classification_report(y_class_test, y_rf_pred, zero_division=0))

Random Forest Classificatio Results
Accuracy: 0.6456666666666667
Precision: 0.25763256561328335
Recall: 0.2105954465849387
F1 Score: 0.2317513852083835
ROC-AUC: 0.4912793206802064

Confusion Matrix:
[[5330 1386]
 [1803  481]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.79      0.77      6716
           1       0.26      0.21      0.23      2284

    accuracy                           0.65      9000
   macro avg       0.50      0.50      0.50      9000
weighted avg       0.62      0.65      0.63      9000



#### Findings

The Random Forest model has not improved our predictive performance. In fact, its results are worse than Logistic Regression on most measures.

However, the comparison gives us useful information about the dataset and the features we have engineered.

**Comparing the three models**

|Metric	|Baseline	|Logistic Regression	|Random Forest|
|---|---:|---:|---:|
|Accuracy	|74.62%	|74.62%	|64.57%|
|Precision	|0.00	|0.00	|25.76%|
|Recall	|0.00	|0.00	|21.06%|
|F1-score	|0.00	|0.00	|23.18%|
|ROC-AUC	|—	|0.499	|0.491|

Random Forest has at least identified some purchasers, unlike Logistic Regression at its default threshold.

But considering what it has achieved:
- It correctly identified only 481 of the 2,284 actual purchasers.
- It missed 1,803 actual purchasers.
- Of the 1,867 customers it classified as purchasers, only 481 actually purchased.

Its precision is only 25.76%, meaning approximately three out of every four customers it identifies as potential purchasers will not purchase.

Most importantly, its ROC-AUC is 0.491, which is below random discrimination.

#### What we learn

There are two possible explanations worth investigating:

1. Limited predictive information: Our historical customer features may not contain enough information to distinguish future purchasers from non-purchasers.
2. Temporal behaviour: Customer purchasing behaviour may have changed between the historical training period and the 2024 testing period.

The second possibility is particularly important because we deliberately used chronological validation rather than randomly splitting observations.

We want to investigate before drawing conclusions.

#### Examining performance by observation date

Keeping in mind that our testing dataset contains three observation dates:
- 31 March 2024
- 30 June 2024
- 30 September 2024

We want to establish whether the model performs consistently across these dates or whether one period is responsible for most of the poor performance.

In [61]:
# Combine actual outcomes, predictions and probabilities
classification_results = pd.DataFrame({
    "snapshot_date": modeling_data.loc[test_mask, "snapshot_date"].values,
    "actual": y_class_test.values,
    "predicted": y_rf_pred,
    "probability": y_rf_prob
})

# Evaluate Random Forest performance by observation date
for snapshot, group in classification_results.groupby("snapshot_date"):

    print(f"\nSnapshot Date: {snapshot.date()}")

    print("Observations:", len(group))

    print(
        "Actual Purchase Rate:",
        round(group["actual"].mean() * 100, 2), "%"
    )

    print(
        "Predicted Purchase Rate:",
        round(group["predicted"].mean() * 100, 2), "%"
    )

    print(
        "Accuracy:",
        round(accuracy_score(group["actual"], group["predicted"]), 4)
    )

    print(
        "Precision:",
        round(precision_score(
            group["actual"], group["predicted"], zero_division=0
        ), 4)
    )

    print(
        "Recall:",
        round(recall_score(
            group["actual"], group["predicted"], zero_division=0
        ), 4)
    )

    print(
        "ROC-AUC:",
        round(roc_auc_score(
            group["actual"], group["probability"]
        ), 4)
    )


Snapshot Date: 2024-03-31
Observations: 3000
Actual Purchase Rate: 25.1 %
Predicted Purchase Rate: 17.57 %
Accuracy: 0.6593
Precision: 0.2448
Recall: 0.1713
ROC-AUC: 0.4973

Snapshot Date: 2024-06-30
Observations: 3000
Actual Purchase Rate: 24.93 %
Predicted Purchase Rate: 20.87 %
Accuracy: 0.648
Precision: 0.254
Recall: 0.2126
ROC-AUC: 0.4857

Snapshot Date: 2024-09-30
Observations: 3000
Actual Purchase Rate: 26.1 %
Predicted Purchase Rate: 23.8 %
Accuracy: 0.6297
Precision: 0.2703
Recall: 0.2465
ROC-AUC: 0.4918


#### Findings

Random Forest is consistently performing poorly across all three testing periods. This is not simply a problem with one particular quarter.

Let's examine the evidence.

1. Performance across the three observation dates

|Metric	|March 2024	|June 2024	|September 2024|
|---|---:|---:|---:|
|Actual purchase rate	|25.10%	|24.93%	|26.10%|
|Predicted purchase rate	|17.57%	|20.87%	|23.80%|
|Accuracy	|65.93%	|64.80%	|62.97%|
|Precision	|24.48%	|25.40%	|27.03%|
|Recall	|17.13%	|21.26%	|24.65%|
|ROC-AUC	|0.4973	|0.4857	|0.4918|


Key observations:

- **Purchase-rate estimation**: The model consistently underestimates the proportion of customers who will purchase.
- **Recall**: It identifies fewer than one in four actual purchasers, even in September.
- **Precision**: Its predictions are only marginally better than selecting customers randomly at the observed purchase rate.
- **ROC-AUC**: All three results remain below 0.5.

2. Conclusion we can draw

We have tested two fundamentally different algorithms:

- Logistic Regression: struggles to distinguish purchasers from non-purchasers.
- Random Forest: also struggles, despite its ability to learn nonlinear relationships.

This suggests that our current historical features may have limited predictive power for this particular target.

However, I would not yet conclude that the dataset cannot support useful predictions.

There is one important aspect of our feature engineering worth investigating: the historical features are cumulative.

For example, total_spending represents everything a customer has spent since their first recorded purchase up to the observation date.

That is useful, but it doesn't necessarily capture recent changes in purchasing behaviour.

A customer who spent EGP 30,000 two years ago and has since become inactive could still have a high historical spending value.

#### Investigating recent purchasing behavior

Before we introduce another algorithm, we want to examine whether we can improve the information available to our existing models.

We want to add just three historical features:

|Feature	|Meaning|
|---|---|
|`orders_last_90_days`	|Number of completed orders in the 90 days preceding the observation date.|
|`spending_last_90_days`	|Completed-order spending during those 90 days.|
|`orders_last_180_days`	|Number of completed orders in the 180 days preceding the observation date.|


These features could help distinguish historically valuable customers from customers who are currently active.

Importantly, all three use information available on or before the observation date. They do not introduce future information into the predictors.

We want to improve the information supplied to the model before making the model more complicated.

Otherwise, we risk repeatedly testing algorithms against the same limited information.